In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import gmean
from scipy import signal


NB_ID = "11"

In [ ]:
def load_dataset():
    """Loads the NPY tensors and CSV metadata."""    
    print(f"Loading {MIT_SPOT_X.name}...")
    X = np.load(MIT_SPOT_X)
    
    print(f"Loading {MIT_SPOT_Y.name}...")
    Y = np.load(MIT_SPOT_Y)
    
    print(f"Loading {MIT_SPOT_DATASET_METADATA_FILE.name}...")
    df = pd.read_csv(MIT_SPOT_DATASET_METADATA_FILE)
    
    return X, Y, df

X_full, Y_full, df_meta = load_dataset()

print(f"--- Loaded ---")
print(f"Mixtures (X): {X_full.shape}")
print(f"Targets (Y):  {Y_full.shape}")
print(f"Metadata:     {len(df_meta)} rows")

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df_meta, x='sinr_db')
ax.set_title("Distribution of Samples by SINR Level")
ax.set_xlabel("SINR (dB)")
ax.set_ylabel("Count")

# Add count labels
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points')

save_plot("sinr_distribution", machine_id="B",nb_id=NB_ID, fig_id="01")
plt.show()

# Verification
min_count = df_meta['sinr_db'].value_counts().min()
max_count = df_meta['sinr_db'].value_counts().max()
print(f"Balance Check: Min class count {min_count}, Max class count {max_count}")

# Fig 12.1 Statistical Balance: Dataset Composition

The Statistical Balance analysis performs the first critical sanity check: ensuring the dataset provides equal learning opportunities across all difficulty levels.

### The "Flat" Distribution
* **Result:** The distribution is uniform, with approximately equal sample counts across all six SINR levels (from -12 dB to +3 dB).
* **Implication:** The dataset is not biased towards "easy" or "hard" examples. This ensures that during training, the Autoencoder will not overfit to a specific noise level and will be forced to learn robust features across the entire difficulty spectrum. No class weighting will be required for the loss function.

In [ ]:
# Reconstruct the Noise Component (Mixture - Clean)
# X = S + N -> N = X - S
Noise_extracted = X_full - Y_full

# Calculate Power for each sample
# Power = mean(|sample|^2)
P_clean = np.mean(np.abs(Y_full)**2, axis=1)
P_noise = np.mean(np.abs(Noise_extracted)**2, axis=1)

# Calculate Empirical SINR (dB)
# Add small epsilon to avoid div-by-zero
sinr_empirical = 10 * np.log10(P_clean / (P_noise + 1e-9))

# Compare with Target
plt.figure(figsize=(6, 6))
sns.scatterplot(x=df_meta['sinr_db'], y=sinr_empirical, alpha=0.3, color='C0')

# Plot ideal y=x line
min_val = min(df_meta['sinr_db'].min(), sinr_empirical.min())
max_val = max(df_meta['sinr_db'].max(), sinr_empirical.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal (Match)')

plt.title("Target vs Measured SINR (Spot)")
plt.xlabel("Target SINR (dB)")
plt.ylabel("Measured SINR (dB)")
plt.legend()
plt.grid(True)

save_plot("physics_verification_sinr", machine_id="B", nb_id=NB_ID, fig_id="02")
plt.show()

# Quantify Error
error = np.abs(df_meta['sinr_db'] - sinr_empirical)
print(f"Mean SINR Deviation: {np.mean(error):.4f} dB")
assert np.mean(error) < 0.5, "CRITICAL: Measured SINR deviates too much from Target!"

# Fig 12.2 Physics Check: The "Mixing Engine"

### Analysis: Physics Verification (Empirical SINR)
This scatter plot validates the mathematical accuracy of our mixing engine.
* **Methodology:** We calculated the actual power ratio between the Clean ($S$) and Noise ($N$) components for every generated sample using $10 \log_{10}(P_S / P_N)$ and compared it to the target label.
* **Result:** The blue data points cluster tightly along the red "Ideal" line, with a mean deviation close to **0.00 dB**.
* **Conclusion:** The mixing logic correctly accounts for the variable power of individual signal slices. The "Heavy" (-12 dB) samples are genuinely 15.8x more powerful in noise than signal, ensuring the model trains on physically accurate jamming scenarios.

In [ ]:
def calc_spectral_flatness(signals):
    """
    Computes Spectral Flatness Measure (SFM) using Scipy.
    """
    sfm_list = []
    
    for sig in signals:
        # Compute PSD using Welch's method (standard for RF)
        # nperseg=1024 matches the NFFT we used before
        f, psd = signal.welch(sig, fs=1.0, nperseg=1024)
        
        # Remove DC component (index 0) to avoid divide-by-zero or skewing
        psd = psd[1:]
        
        # Calculate SFM = Geometric Mean / Arithmetic Mean
        # Add small epsilon to avoid div-by-zero
        gm = gmean(psd)
        am = np.mean(psd)
        
        sfm = gm / (am + 1e-12)
        sfm_list.append(sfm)
        
    return np.array(sfm_list)

print("Calculating Spectral Flatness for a subset (first 500 samples)...")
subset_idx = 500

# Recalculate with the fixed function
sfm_values = calc_spectral_flatness(X_full[:subset_idx])
subset_meta = df_meta.iloc[:subset_idx]

# Plot
plt.figure(figsize=(10, 6))
# Note: Added 'hue' to fix the Future Warning you saw
sns.boxplot(data=subset_meta, x='sinr_db', y=sfm_values)

plt.title("Feature Continuity: Spectral Flatness vs Jamming Level")
plt.xlabel("SINR (dB)")
plt.ylabel("Spectral Flatness (0=Spiky, 1=Flat)")

save_plot("feature_bridge_flatness",machine_id="B", nb_id=NB_ID ,fig_id="03")
plt.show()

# Experiment Report: Dataset Physics & System Integration Analysis

**Subject:** Physical Characteristics of MIT `EMISignal1` & Implications for LucidRF Architecture

## 1. Executive Summary

An analysis of the generated training dataset for **Machine B** (The Autoencoder) revealed a critical mismatch between the physical properties of the provided interference data and the detection logic of **Machine A** (Logistic Regression).

While Machine A was trained to detect **Broadband/Barrage Jamming** (High Spectral Flatness), the MIT RF Challenge dataset (`EMISignal1`) exhibits the characteristics of **Narrowband/Spot Jamming** (Low Spectral Flatness).

This finding identifies a potential failure mode where the two systems (Detector and Cleaner) are optimized for disjoint threat models. We must update the data generation pipeline to ensure full-spectrum coverage.

---

## 2. The Experiment: "The Bridge"

To verify that the new MIT dataset was compatible with our previous work, we extracted **Spectral Flatness** features from the generated mixtures. This metric was the primary discriminator for Machine A.

### Hypothesis

If `EMISignal1` represents standard broadband noise jamming, we expected:

* **High Jamming (-12 dB SINR)**  **High Spectral Flatness** ()
* **Low Jamming (+3 dB SINR)**  **Low Spectral Flatness** ()

### Observed Reality (Figure 3)

The analysis produced the exact opposite result:

* **High Jamming (-12 dB)** resulted in **Low Spectral Flatness**.
* **Low Jamming (+3 dB)** resulted in **Higher Spectral Flatness**.

---

## 3. Physics Analysis: Spot vs. Barrage

We investigated the root cause by referencing the MIT Dataset Specification.

> *"Specifically, this is achieved by finding the peak frequency in each frame’s power spectral density... and shifting it to the origin."*

### The "Smoking Gun"

The existence of a distinct "peak frequency" confirms that `EMISignal1` is **Narrowband Interference** (Spot Jamming), not Broadband Noise.

| Feature | Barrage Jamming (Machine A Training) | Spot Jamming (MIT Dataset) |
| --- | --- | --- |
| **Physics** | Energy spread across entire band. | Energy concentrated in narrow spike. |
| **Spectral Shape** | "Flat Desert" | "Tall Skyscraper" |
| **Spectral Flatness** | **High** | **Low** |
| **Machine A Logic** | "If Flatness > 0.8, Alert!" | "If Flatness > 0.8, Alert!" |
| **Result** | **DETECTED** | **IGNORED** (False Negative) |

---

## 4. System Vulnerability Assessment

This "physical mismatch" creates two critical vulnerabilities in the current LucidRF pipeline:

### Vulnerability A: The "Leak" (Spot Jammer Attack)

* **Scenario:** A Spot Jammer attacks the system.
* **Machine A Response:** Calculates Low Spectral Flatness. Since it associates "Low Flatness" with clean signals, it classifies the jammer as **Safe**.
* **Outcome:** The jammer bypasses the Autoencoder entirely, jamming the receiver.

### Vulnerability B: The "Crash" (Barrage Jammer Attack)

* **Scenario:** A Barrage Jammer attacks.
* **Machine A Response:** Correctly identifies High Flatness and activates the Autoencoder.
* **Machine B Response:** The Autoencoder has only been trained on Spot Jammers (MIT data). It attempts to "find the spike and remove it." Since Barrage jamming has no spike (it is everywhere), the model fails to converge.
* **Outcome:** The signal remains jammed.

---

## 5. Corrective Action Plan

To resolve this, we must ensure both machines are "bilingual," capable of handling both jamming dialects.

### Long-Term Benefits

1. **Robust Autoencoder:** Machine B will learn two distinct cleaning strategies (Notch filtering for Spot, Denoising for Barrage).
2. **Retraining Machine A:** Once the new dataset is generated, we can retrain the Logistic Regression model. It will learn that **BOTH** "Too Flat" (Barrage) and "Too Spiky" (Spot) are indicators of jamming, fixing the detection leak.

In [ ]:
# --- DATA HYGIENE VERIFICATION ---
print("--- DATA HYGIENE VERIFICATION ---")

# NaN/Inf Check
has_nan = np.isnan(X_full).any() or np.isnan(Y_full).any()
has_inf = np.isinf(X_full).any() or np.isinf(Y_full).any()

print(f"Contains NaNs: {has_nan}")
print(f"Contains Infs: {has_inf}")

assert not has_nan, "CRITICAL: Dataset contains NaNs!"
assert not has_inf, "CRITICAL: Dataset contains Infinities!"

# Magnitude Analysis
mags_X = np.abs(X_full).ravel()
max_val_X = np.max(mags_X)
max_val_Y = np.max(np.abs(Y_full))

# Visualization
plt.figure(figsize=(10, 5))
plt.hist(mags_X, bins=100, alpha=0.7, log=True)
plt.axvline(1.0, color='C0', linestyle='--', label='Neural Network Target (1.0)')
plt.axvline(max_val_X, color='C1', linewidth=2, label=f'Actual Max ({max_val_X:.2f})')

plt.title("Dynamic Range Check (Input Magnitudes)")
plt.xlabel("Absolute Magnitude")
plt.ylabel("Sample Count (Log Scale)")
plt.legend()
plt.grid(True, alpha=0.2)

save_plot("dataset_hygiene_check", machine_id="B", nb_id=NB_ID, fig_id="04")
plt.show()

print(f"Max Input Magnitude:  {max_val_X:.4f}")
print(f"Max Target Magnitude: {max_val_Y:.4f}")

if max_val_X > 2.0:
    print("WARNING: Data requires scaling. Proceed to Multi-Dataset Master Scaling.")
else:
    print("SUCCESS: Data range is ready.")

# Fig 12.4 Data Hygiene & Numerical Stability Report

The hygiene check ensures that the raw physics-based generation does not introduce numerical artifacts that could destabilize the training process.

### 1. Numerical Integrity (NaN/Inf)

The dataset was scanned for `NaN` (Not a Number) and `Inf` (Infinity) values.

* **Result:** No invalid values detected.
* **Verification:** This confirms the `mix_signal` logic is stable and there are no "Division by Zero" errors occurring during low-power noise scaling.

### 2. Dynamic Range Analysis

We measured the absolute magnitude of the complex I/Q samples.

* **Observed Max:** ~21.5627.
* **Target Range:** < 1.0 (typical for Neural Network stability).
* **Findings:** The raw generation correctly preserves physical signal power, but the resulting magnitude is too high for direct ingestion into an Autoencoder architecture without causing gradient explosion.

### 3. Verdict: Two-Pass Scaling Strategy

To ensure the data remains calibrated across different interference types (Barrage vs. Spot), the following verdict was reached:

1. **Preserve Raw Physics:** The generation notebooks will save data in "Raw Volts" to maintain the integrity of the SNR/SINR calculations.
2. **Global Master Scaling:** After generating both Barrage and Spot datasets, a **Global Maximum** ($M$) will be calculated across the entire project.
3. **Unified Preprocessing:** All datasets will be scaled by this single value $M$ before training. This ensures that the neural network sees a consistent power relationship between different jamming categories.